In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
from pathlib import Path
from IPython.display import Audio
import IPython.display as ipd
from scipy.io import wavfile
import tempfile
import os
import librosa
import pandas as pd
import seaborn as sns
import h5py
import mne
from scipy.stats import zscore
from mne_bids import BIDSPath, read_raw_bids
from matplotlib_venn import venn2,venn2_circles
from tqdm import tqdm

In [2]:
cm = 1/2.54
plt.rcParams['svg.fonttype'] = 'none'

fontdict = dict(fontsize=7)
fontsize = 7

red = '#A9373B'
blue = '#2369BD'
orange = '#CC8963'
green = '#009944'

stg_color = '#20B2AA'
smc_color = '#6A5ACD'
insula_color = '#D4AF37'

reds = sns.light_palette(red, as_cmap=True)
blues = sns.light_palette(blue, as_cmap=True)
oranges = sns.light_palette(orange, as_cmap=True)
greens = sns.light_palette(green, as_cmap=True)

recon_dir = '/cwork/ns458/ECoG_Recon/'
mne.viz.set_3d_backend('notebook')                    # MNE 3D in-notebook static backend
# text svg

Using notebook 3d backend.


In [ ]:
LEXICAL_PTS = ''

In [ ]:
epos = []

pts = BIDSPath(
    root=os.path.join(bids_root, 'derivatives', 'epoch(bipolar)'),
    datatype='epoch(band)(zscore)',
    suffix='highgamma',
    subject=sub,
    extension='.h5',
    check=False
)

for pt in pts.match():
    dc_epo = mne.read_epochs(pt, verbose='error')
    # onset = dc_epo.events[:, 0] / 2048
    # dc_epo.metadata = pd.DataFrame({'onset': onset}, index=dc_epo.selection)
    dc_epo.pick(picks)
    df = dc_epo.to_data_frame(long_format=True, scalings={'seeg': 1}, verbose=False)
    # meta = dc_epo.metadata.reset_index(names='epoch')  # epoch == selection
    # df = df.merge(meta, on='epoch', how='left')

    # drop ch_type column
    df = df.drop(columns=['ch_type'])
    df['phase'] = pt.processing
    df['description'] = pt.description
    df['subject'] = sub
    df['remark'] = df['condition'].str.split('/').str[4:].str.join('/')
    df['condition'] = df['condition'].str.split('/').str[2:4].str.join('/')
    df['lexical'] = df['condition'].str.split('/').str[0]
    
    # Map lexical values: Word -> Yes, Non-Word -> No
    df['lexical'] = df['lexical'].map({'Word': 'Yes', 'Nonword': 'No'})
    
    epos.append(df)

# Concatenate across subjects (optional)
epos = pd.concat(epos, ignore_index=True)
epos.head()